# 002 · Running Tasks Concurrently

> **Syntax › Async** · Python 3.14

Awaiting coroutines one after another runs them sequentially. To get real concurrency you turn coroutines into **tasks** and let the event loop interleave them. This notebook maps the `asyncio` toolbox onto the `Task` APIs you know from .NET.

**In this notebook you will learn:**

- `asyncio.create_task` vs starting a `Task` in C#
- `asyncio.gather` vs `Task.WhenAll`
- `asyncio.TaskGroup` (structured concurrency), `ExceptionGroup` and `except*`
- timeouts with `asyncio.timeout` / `asyncio.wait_for` vs `CancellationToken`
- cancelling tasks: `CancelledError` vs `OperationCanceledException`
- `asyncio.as_completed` and `asyncio.wait` vs `Task.WhenAny`
- throttling with `asyncio.Semaphore` and running blocking code with `asyncio.to_thread`

## C# mental model

| C# | Python |
|---|---|
| `var t = DoAsync();` (already running) | `t = asyncio.create_task(do())` |
| `await Task.WhenAll(a, b)` | `await asyncio.gather(a, b)` or `asyncio.TaskGroup` |
| `await Task.WhenAny(a, b)` | `asyncio.wait(..., return_when=FIRST_COMPLETED)` / `as_completed` |
| `CancellationToken` passed everywhere | `task.cancel()` injects `CancelledError` at the next `await` |
| `OperationCanceledException` | `asyncio.CancelledError` (a `BaseException`) |
| `new CancellationTokenSource(timeout)` / `.WaitAsync(timeout)` | `async with asyncio.timeout(s)` / `asyncio.wait_for(...)` |
| `AggregateException` | `ExceptionGroup` + `except*` |
| `SemaphoreSlim.WaitAsync()` | `async with asyncio.Semaphore(n)` |
| `Task.Run(() => Blocking())` | `await asyncio.to_thread(blocking)` |

All examples use `asyncio.sleep` to simulate I/O, and top-level `await` (available in Jupyter; scripts use `asyncio.run(main())`, see *Async › 001 · Async and Await*).

In [ ]:
import asyncio
import time


async def fetch(name: str, delay: float) -> str:
    await asyncio.sleep(delay)          # simulate network I/O
    return f"{name} ({delay}s)"


class Timer:
    """Tiny helper: `with Timer(): ...` prints the elapsed time."""
    def __enter__(self):
        self.start = time.perf_counter()
    def __exit__(self, *exc):
        print(f"  elapsed: {time.perf_counter() - self.start:.2f}s")

## Sequential vs concurrent

Awaiting one coroutine after another takes the **sum** of the delays:

In [ ]:
with Timer():
    a = await fetch("a", 0.2)
    b = await fetch("b", 0.2)
    print(a, b)

## `asyncio.create_task`: start it now

`create_task` wraps a coroutine in a `Task` and **schedules it on the loop**. It starts running as soon as the current coroutine yields (at its next `await`). This is the closest equivalent of calling a C# async method without awaiting it yet.

```csharp
// C#
var a = FetchAsync("a", 200);   // both start immediately
var b = FetchAsync("b", 200);
Console.WriteLine($"{await a} {await b}");   // ~200 ms total
```

In [ ]:
with Timer():
    task_a = asyncio.create_task(fetch("a", 0.2))
    task_b = asyncio.create_task(fetch("b", 0.2))
    print(type(task_a).__name__, task_a.done())   # Task False: not finished yet
    print(await task_a, await task_b)             # ~0.2s, not 0.4s

Unlike a bare coroutine, a `Task` **can** be awaited several times (it caches its result like a C# `Task`) and exposes `done()`, `result()`, `cancel()` and `cancelled()`.

⚠️ **Gotcha:** the event loop only keeps a **weak reference** to tasks. A fire-and-forget `asyncio.create_task(...)` whose result you never store can be garbage-collected mid-flight. Keep a reference (e.g. in a set), or better, use a `TaskGroup` (below).

## `asyncio.gather` vs `Task.WhenAll`

`gather` runs several awaitables concurrently and returns their results **in argument order**, regardless of which finished first.

```csharp
// C#
string[] results = await Task.WhenAll(FetchAsync("a", 300), FetchAsync("b", 100), FetchAsync("c", 200));
```

In [ ]:
with Timer():
    results = await asyncio.gather(
        fetch("a", 0.3),
        fetch("b", 0.1),
        fetch("c", 0.2),
    )
    print(results)           # in argument order, total ~0.3s

With a dynamic number of coroutines, unpack a list (or generator) with `*` *(see Functions › 003 · Args and Kwargs)*:

In [ ]:
names = ["alice", "bob", "carol"]
users = await asyncio.gather(*(fetch(n, 0.1) for n in names))
print(users)

By default, the **first exception** propagates out of `gather`, but the other tasks are **not cancelled**: they keep running in the background. With `return_exceptions=True`, exceptions are returned as values in the result list instead, similar to inspecting each task after `Task.WhenAll` throws.

In [ ]:
async def fail(message: str, delay: float):
    await asyncio.sleep(delay)
    raise ValueError(message)


try:
    await asyncio.gather(fetch("ok", 0.1), fail("boom", 0.05))
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

results = await asyncio.gather(fetch("ok", 0.1), fail("boom", 0.05), return_exceptions=True)
print(results)

## `asyncio.TaskGroup`: structured concurrency (3.11+)

`TaskGroup` is the modern, safer way to run tasks together. Every task started in the `async with` block is guaranteed to be finished when the block exits. If any task fails, the **remaining tasks are cancelled** and all errors are raised together as an `ExceptionGroup`. C# has no built-in equivalent; the closest is `Task.WhenAll` plus a linked `CancellationTokenSource`.

In [ ]:
with Timer():
    async with asyncio.TaskGroup() as tg:
        t1 = tg.create_task(fetch("orders", 0.2))
        t2 = tg.create_task(fetch("invoices", 0.1))
    # here both tasks are guaranteed to be done
    print(t1.result(), "|", t2.result())

When tasks fail, catch the errors with `except*`, which matches exceptions **inside** an `ExceptionGroup` by type. It is like handling `AggregateException.InnerExceptions`, but with the type filtering built in. Each `except*` clause receives a sub-group containing only the matching exceptions.

In [ ]:
async def slow_ok():
    try:
        await asyncio.sleep(0.5)
        return "finished"
    except asyncio.CancelledError:
        print("  slow_ok was cancelled because a sibling failed")
        raise                                   # always re-raise CancelledError


try:
    async with asyncio.TaskGroup() as tg:
        tg.create_task(slow_ok())
        tg.create_task(fail("bad input", 0.05))
        tg.create_task(fail("also bad", 0.05))
except* ValueError as group:
    print(f"{type(group).__name__}: {[str(e) for e in group.exceptions]}")

`except*` can have several clauses for different types, and each one runs if its type is present in the group. (Python 3.14 also allows `except A, B:` without parentheses when catching several types; the parenthesised `except (A, B):` form works in every version and is what we use *(see Error Handling › 001 · try / except / finally)*.)

| | `gather` | `TaskGroup` |
|---|---|---|
| Results | list, argument order | `task.result()` on each task |
| On failure | first exception raised, others keep running | others cancelled, `ExceptionGroup` raised |
| Collect errors as values | `return_exceptions=True` | no, use `except*` |
| Recommended for new code | when you want the results list | yes, for most cases |

## Timeouts: `asyncio.timeout` and `asyncio.wait_for`

`async with asyncio.timeout(seconds)` (3.11+) cancels whatever is running inside the block when time runs out, and converts the cancellation into a `TimeoutError`. `asyncio.wait_for(aw, seconds)` does the same for a single awaitable.

```csharp
// C#
using var cts = new CancellationTokenSource(TimeSpan.FromMilliseconds(100));
try { await FetchAsync("slow", 500, cts.Token); }
catch (OperationCanceledException) { Console.WriteLine("timed out"); }

// or, for a single task (.NET 6+)
await FetchAsync("slow", 500).WaitAsync(TimeSpan.FromMilliseconds(100));
```

In [ ]:
try:
    async with asyncio.timeout(0.1):
        await fetch("slow", 0.5)
except TimeoutError as e:
    print(f"{type(e).__name__} from asyncio.timeout")

try:
    await asyncio.wait_for(fetch("slow", 0.5), timeout=0.1)
except TimeoutError as e:
    print(f"{type(e).__name__} from asyncio.wait_for")

print(await asyncio.wait_for(fetch("fast", 0.05), timeout=0.1))

The big design difference: in .NET, cancellation is **cooperative** and you must thread a `CancellationToken` through every method. In Python, you pass nothing: the event loop cancels the task by throwing `CancelledError` into it at its current `await`.

## Cancellation: `task.cancel()`

`task.cancel()` requests cancellation; the task receives `asyncio.CancelledError` at its next `await`. The task can run cleanup in `except`/`finally`, and awaiting a cancelled task raises `CancelledError` in the awaiter, like awaiting a cancelled C# task throws `TaskCanceledException`.

In [ ]:
async def worker():
    try:
        for i in range(10):
            print(f"  working {i}")
            await asyncio.sleep(0.05)
    finally:
        print("  worker cleanup")          # runs on cancellation too


task = asyncio.create_task(worker())
await asyncio.sleep(0.12)                   # let it do a little work
task.cancel()

try:
    await task
except asyncio.CancelledError:
    print("task cancelled:", task.cancelled())

⚠️ **Gotcha:** `CancelledError` derives from `BaseException`, **not** `Exception` (since Python 3.8), so a broad `except Exception:` does not swallow it. If you do catch `CancelledError` (e.g. for logging), **re-raise it**, otherwise cancellation, `TaskGroup` and `timeout()` stop working correctly. In C#, swallowing `OperationCanceledException` is similarly a bug.

In [ ]:
print(issubclass(asyncio.CancelledError, Exception))       # False
print(issubclass(asyncio.CancelledError, BaseException))   # True
print(issubclass(asyncio.TimeoutError, TimeoutError))      # True: same class since 3.11

## Completion order: `as_completed` and `asyncio.wait`

`gather` waits for **all** results in argument order. To process results **as soon as each one finishes**, use `asyncio.as_completed`. Since 3.13 it supports `async for`, yielding the original tasks in completion order.

```csharp
// C#: process in completion order
var pending = new List<Task<string>> { FetchAsync("a", 300), FetchAsync("b", 100), FetchAsync("c", 200) };
while (pending.Count > 0)
{
    var done = await Task.WhenAny(pending);
    pending.Remove(done);
    Console.WriteLine(await done);
}
```

In [ ]:
tasks = [asyncio.create_task(fetch(n, d)) for n, d in [("a", 0.3), ("b", 0.1), ("c", 0.2)]]

async for finished in asyncio.as_completed(tasks):
    print("done:", await finished)          # b, c, a

For a literal `Task.WhenAny`, use `asyncio.wait` with `return_when=FIRST_COMPLETED`. It returns two sets, `done` and `pending`; remember to cancel the losers if you no longer need them.

In [ ]:
tasks = [asyncio.create_task(fetch(f"mirror-{i}", d)) for i, d in enumerate([0.3, 0.1, 0.2])]

done, pending = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)
print("winner:", done.pop().result())
for t in pending:
    t.cancel()                              # first response wins, cancel the rest
print("cancelled", len(pending), "slower tasks")

## Throttling with `asyncio.Semaphore`

Starting 1,000 requests at once can overwhelm a server. A semaphore limits how many tasks are inside a block at the same time, exactly like `SemaphoreSlim` in C#, but `async with` replaces the `WaitAsync()`/`try`/`finally`/`Release()` dance.

```csharp
// C#
var gate = new SemaphoreSlim(2);
await gate.WaitAsync();
try { await DownloadAsync(url); }
finally { gate.Release(); }
```

In [ ]:
gate = asyncio.Semaphore(2)
active = 0
peak = 0


async def download(item: int) -> int:
    global active, peak
    async with gate:                        # at most 2 at a time
        active += 1
        peak = max(peak, active)
        await asyncio.sleep(0.1)
        active -= 1
        return item


with Timer():
    results = await asyncio.gather(*(download(i) for i in range(6)))
    print(results, "| peak concurrency:", peak)   # 6 items / 2 slots -> ~0.3s

## Blocking code: `asyncio.to_thread`

Sometimes you must call a blocking function (a synchronous library, `time.sleep`, file I/O). Running it directly freezes the event loop *(see Async › 001 · Async and Await)*. `asyncio.to_thread` runs it in a worker thread and gives you an awaitable, much like `await Task.Run(() => Blocking())` in C#.

In [ ]:
import threading


def blocking_report(name: str) -> str:
    time.sleep(0.2)                         # a blocking call we cannot change
    return f"{name} built on {threading.current_thread().name}"


with Timer():
    results = await asyncio.gather(
        asyncio.to_thread(blocking_report, "report-1"),
        asyncio.to_thread(blocking_report, "report-2"),
        fetch("api", 0.2),                  # async work keeps running alongside
    )
    for r in results:
        print(" ", r)

Threads help with **blocking I/O** because blocking calls release the GIL while they wait. For **CPU-bound** pure-Python work on a standard (non-free-threaded) build, threads won't run in parallel; use `loop.run_in_executor` with a `concurrent.futures.ProcessPoolExecutor` instead.

## Summary

| Task | C# | Python |
|---|---|---|
| Start a task now | `var t = FooAsync();` | `t = asyncio.create_task(foo())` |
| Wait for all | `Task.WhenAll(...)` | `asyncio.gather(...)` / `asyncio.TaskGroup()` |
| Structured concurrency | n/a | `async with asyncio.TaskGroup() as tg:` |
| Multiple failures | `AggregateException` | `ExceptionGroup` + `except*` |
| Wait for first | `Task.WhenAny(...)` | `asyncio.wait(..., return_when=FIRST_COMPLETED)` |
| In completion order | loop over `Task.WhenAny` | `async for t in asyncio.as_completed(...)` |
| Timeout | `CancellationTokenSource(timeout)` / `.WaitAsync(t)` | `asyncio.timeout(s)` / `asyncio.wait_for(aw, s)` |
| Cancel | `cts.Cancel()` + token checks | `task.cancel()` |
| Cancellation exception | `OperationCanceledException` | `asyncio.CancelledError` (`BaseException`) |
| Throttle | `SemaphoreSlim` | `asyncio.Semaphore` |
| Offload blocking call | `Task.Run(...)` | `asyncio.to_thread(...)` |

## Exercises

1. Use `asyncio.gather` to fetch `["a", "b", "c", "d"]` with delays `0.1, 0.2, 0.1, 0.2` and confirm the total time is about 0.2 s.
2. Rewrite exercise 1 with `asyncio.TaskGroup` and print each task's result.
3. Write `fetch_with_fallback(name)` that tries `fetch(name, 0.5)` with a 0.1 s timeout and returns `"cached"` on timeout.
4. Download 10 items with `download`-style throttling so that at most 3 run at the same time, processing results in completion order.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import asyncio, time

# 1
start = time.perf_counter()
results = await asyncio.gather(*(fetch(n, d) for n, d in zip("abcd", [0.1, 0.2, 0.1, 0.2])))
print(results, f"{time.perf_counter() - start:.2f}s")

# 2
async with asyncio.TaskGroup() as tg:
    tasks = [tg.create_task(fetch(n, d)) for n, d in zip("abcd", [0.1, 0.2, 0.1, 0.2])]
print([t.result() for t in tasks])

# 3
async def fetch_with_fallback(name):
    try:
        async with asyncio.timeout(0.1):
            return await fetch(name, 0.5)
    except TimeoutError:
        return "cached"

print(await fetch_with_fallback("profile"))

# 4
gate = asyncio.Semaphore(3)

async def limited(i):
    async with gate:
        await asyncio.sleep(0.05)
        return i

tasks = [asyncio.create_task(limited(i)) for i in range(10)]
async for t in asyncio.as_completed(tasks):
    print(await t, end=" ")
```

</details>